# Phase 3: ScreenSpot -- the public benchmark (Kaggle, resumable)

Our v3 splits say fine-tuning took click accuracy from 1.5% to 46.5% on Windows apps the model never saw.
**ScreenSpot** (SeeClick, 2024) is other people's screenshots: iOS/Android, macOS/Windows and four websites,
1,272 instructions written by people. It answers what our splits can't:
1. a number comparable to what others report, and
2. does training on 10 Windows apps help, hurt, or do nothing on GUIs it was never near?

**Fixed before any ScreenSpot number is seen** (see `src/computeruse/eval/screenspot.py`):
the adapter is the one already chosen on our dev split (`eval/selection.json`); instructions are used verbatim;
images above the training pixel budget (1568x882) are scaled down, smaller ones untouched; both arms see the same pixels.
Nothing about ScreenSpot feeds back into any choice.

Every scored example goes to `screenspot/results/<arm>.jsonl` on the Hub every few minutes; a rerun resumes.

## Setup (same as the other notebooks)
Secret `HF_TOKEN` attached, **GPU T4 x2**, Internet **On**. No Kaggle dataset needed: ScreenSpot downloads from the Hub.

## Run: Save Version -> "Save & Run All (Commit)" with `RUN_MODE = "eval"` (~2 h)
It checks itself after the first 20 answers and stops if they don't parse. `RUN_MODE = "smoke"` scores 24 examples (~5 min).

In [ ]:
# Cell 1 -- settings. Runs before ANY torch/transformers import, on purpose.
import os
import time

SESSION_START = time.time()

RUN_MODE = "eval"                 # "eval" (all 1,272) or "smoke" (24 examples)
HF_REPO_NAME = "computer-use-grounder"   # the training repo: the adapter comes from it, results go back to it
CODE_REF = "dafd44bb18f41cc53e4d94860a8a8e24bc2e4ced"           # exact commit of rudranaresh0201/computer-use

assert RUN_MODE in ("smoke", "eval"), RUN_MODE
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"RUN_MODE={RUN_MODE}  CODE_REF={CODE_REF}")

In [ ]:
# Cell 2 -- the GPU the model was trained and evaluated on.
import subprocess

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0, "No GPU attached. Session options -> Accelerator -> GPU T4 x2."
assert "T4" in smi.stdout, "Expected a Tesla T4 (Session options -> Accelerator -> GPU T4 x2)."

In [ ]:
# Cell 3 -- the token must work in THIS committed session and be able to write results.
import requests
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Could not read the HF_TOKEN secret. Add-ons -> Secrets: add HF_TOKEN, tick it for this notebook.") from e
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN secret is empty or not a Hugging Face token"
r = requests.get("https://huggingface.co/api/whoami-v2", headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=30)
assert r.status_code == 200, f"Hugging Face rejected the token (HTTP {r.status_code}): {r.text[:200]}"
me = r.json()
assert me.get("auth", {}).get("accessToken", {}).get("role") != "read", "HF_TOKEN is READ-only; results need write."
HF_REPO = f"{me['name']}/{HF_REPO_NAME}"
PREFIX = "screenspot" if RUN_MODE == "eval" else f"screenspot-smoke-{CODE_REF[:7]}"
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"token OK for {me['name']}; adapter from {HF_REPO}, results to {HF_REPO}/{PREFIX}/")

In [ ]:
# Cell 4 -- the exact code version, and its pinned dependencies.
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/computer-use")
if not REPO_DIR.is_dir():
    !git clone https://github.com/rudranaresh0201/computer-use.git {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin
!git -C {REPO_DIR} checkout --quiet {CODE_REF}
!git -C {REPO_DIR} log -1 --format="evaluating with commit %H (%s)"
!{sys.executable} -m pip install --quiet -e "{REPO_DIR}[training]"
!{sys.executable} -m pip uninstall -y --quiet torchao

In [ ]:
# Cell 5 -- imports at the pinned versions, and the adapter our own dev split chose.
import importlib
import json

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()

import pyarrow  # noqa: F401  (reads the ScreenSpot shards)
import torch
import transformers

from computeruse.training.hub_checkpoint import HubSync

print(f"torch {torch.__version__} | transformers {transformers.__version__}")
assert transformers.__version__ == "5.0.0", "pinned transformers did not load -- restart and run from the top"
assert torch.cuda.is_available()

sync = HubSync(HF_REPO, token=HF_TOKEN)
print(f"Hub write check OK as {sync.preflight()}")
STORE_DIR = Path("/kaggle/working/screenspot_store")
assert sync.fetch(["eval/selection.json"], STORE_DIR), "no eval/selection.json -- run the v3 eval notebook first"
CHOSEN = json.loads((STORE_DIR / "eval" / "selection.json").read_text())["chosen"]
print(f"adapter chosen on our dev split: {CHOSEN}")

In [ ]:
# Cell 6 -- ScreenSpot itself (~600 MB of parquet from the Hub).
from collections import Counter

from computeruse.eval import screenspot

ITEMS = screenspot.load(cache_dir=Path("/kaggle/working/hf_cache"))
print(f"ScreenSpot: {len(ITEMS)} examples")
print("by slice: ", dict(sorted(Counter(i.slice for i in ITEMS).items())))
print("by source:", dict(sorted(Counter(i.data_source for i in ITEMS).items())))
assert len(ITEMS) == screenspot.EXPECTED_ROWS, f"expected {screenspot.EXPECTED_ROWS} rows"
if RUN_MODE == "smoke":
    by_slice = {}
    for item in ITEMS:
        by_slice.setdefault(item.slice, []).append(item)
    ITEMS = [i for group in by_slice.values() for i in group[:4]]  # 4 per slice
    print(f"smoke: {len(ITEMS)} examples")

In [ ]:
# Cell 7 -- base model + the one chosen adapter.
from transformers import AutoModelForImageTextToText, AutoProcessor

from computeruse.eval.ablation import adapter_path
from computeruse.eval.resumable import ResultStore
from computeruse.eval.vlm_grounder import MODEL_ID, ZERO_SHOT, arm_active, load_adapters
from computeruse.training.train_lora import resolve_precision, torch_dtype_for

ADAPTER_ROOT = Path("/kaggle/working/adapters")
folder = "final" if CHOSEN == "final" else f"adapters/{CHOSEN}"
assert sync.fetch([f"{folder}/adapter_*"], ADAPTER_ROOT), f"adapter {CHOSEN} not found on the Hub"

precision = resolve_precision()
processor = AutoProcessor.from_pretrained(MODEL_ID)
base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch_dtype_for(precision), attn_implementation="sdpa"
).to("cuda")
model = load_adapters(base, {CHOSEN: adapter_path(CHOSEN, ADAPTER_ROOT)})
del base  # the plain base model is now only arm_active(model, ZERO_SHOT)
print(f"{MODEL_ID} in {precision} + {CHOSEN}; GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")

store = ResultStore(sync, PREFIX, STORE_DIR, flush_every_seconds=300)
print(f"results already on the Hub under {PREFIX}/: {store.pull()}")


def score(item):
    return screenspot.score_item(model, processor, item)

In [ ]:
# Cell 8 -- score both arms. The fine-tuned model goes first, and stops the run by itself if its
# first 20 answers don't parse (it parsed 100% on our own splits) -- no hours burned on a broken setup.
from computeruse.eval.resumable import score_arm
from computeruse.eval.vlm_grounder import format_counts, summarize_diagnostics

ITEM_BY_ID = {i.id: i for i in ITEMS}

with arm_active(model, CHOSEN), torch.no_grad():
    first = score_arm(store, CHOSEN, ITEMS[:20], score)
parse_rate = summarize_diagnostics(first)["parse_rate"]
print(f"\nsanity check -- {CHOSEN} parse rate on the first 20: {parse_rate:.0%}")
for r in first[:6]:
    item = ITEM_BY_ID[r.example_id]
    print(f"  {item.slice:<13} {item.instruction[:40]!r:<44} -> {r.predicted_text!r} hit={r.hit}")
assert parse_rate >= 0.8, "the fine-tuned model's answers are not parsing -- stopping before spending the GPU budget"

RESULTS = {}
t_start = time.time()
for arm in (CHOSEN, ZERO_SHOT):
    with arm_active(model, arm), torch.no_grad():
        RESULTS[arm] = score_arm(store, arm, ITEMS, score)
print("zero-shot answer formats:", format_counts(RESULTS[ZERO_SHOT]))
print(f"scoring took {(time.time() - t_start) / 3600:.1f} h")

In [ ]:
# Cell 9 -- the report: ScreenSpot's usual table, with 95% intervals, and the paired difference.
REPORT = screenspot.report({ZERO_SHOT: RESULTS[ZERO_SHOT], "fine_tuned": RESULTS[CHOSEN]})
REPORT.update({"adapter": CHOSEN, "code_ref": CODE_REF, "examples": len(ITEMS),
               "zero_shot_formats": format_counts(RESULTS[ZERO_SHOT])})
Path("/kaggle/working/screenspot_report.json").write_text(json.dumps(REPORT, indent=2))
store.push_json("report.json", REPORT)

columns = [f"{p}/{t}" for p in screenspot.PLATFORMS for t in screenspot.TYPES] + list(screenspot.PLATFORMS) + ["overall"]


def fmt(cell):
    return "-" if cell["accuracy"] is None else f"{cell['accuracy']:.1%}"


print(f"\nScreenSpot click accuracy (fine_tuned = {CHOSEN})")
print(f"{'arm':<11}" + "".join(f"{c:>15}" for c in columns))
for arm in (ZERO_SHOT, "fine_tuned"):
    print(f"{arm:<11}" + "".join(f"{fmt(REPORT['arms'][arm][c]):>15}" for c in columns))
print(f"{'n':<11}" + "".join(f"{REPORT['arms']['fine_tuned'][c]['n']:>15}" for c in columns))

print("\nfine-tuned minus zero-shot, paired, 95% CI:")
for group, d in REPORT["vs_zero_shot"]["fine_tuned"].items():
    lo, hi = d["ci95"]
    word = "better" if lo > 0 else "worse" if hi < 0 else "no clear difference"
    print(f"  {group:<8} {d['diff']:+.1%} [{lo:+.1%}, {hi:+.1%}]  n={d['n']}  -> {word}")

print("\n" + "=" * 70)
hours = (time.time() - SESSION_START) / 3600
print(("SCREENSPOT SMOKE PASSED" if RUN_MODE == "smoke" else "SCREENSPOT FINISHED") +
      f" in {hours:.1f} h. Report: {HF_REPO}/{PREFIX}/report.json")
print("=" * 70)